In [ ]:
# Cell 1 — Ensure dependencies are available
# timm and librosa are pre-installed on Kaggle; this is a safety net only.
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)


_ensure("timm")
_ensure("librosa")

import librosa
import timm

print(f"timm {timm.__version__}  librosa {librosa.__version__}")

In [ ]:
# Cell 2 — Imports and config
import glob as _glob
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn

# ---------------------------------------------------------------------------
# Audio / spectrogram constants — must match train.py exactly
# ---------------------------------------------------------------------------
SR = 32000
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 1024
FMIN = 20
FMAX = 16000
CLIP_DURATION = 5  # seconds per chunk
CLIP_SAMPLES = SR * CLIP_DURATION

# ImageNet normalisation applied after z-score (matches spec_to_tensor in train.py)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ---------------------------------------------------------------------------
# Paths — auto-detect by searching for key files
# ---------------------------------------------------------------------------

# Competition data: find taxonomy.csv
_tax_matches = _glob.glob("/kaggle/input/**/taxonomy.csv", recursive=True)
if _tax_matches:
    DATA = Path(_tax_matches[0]).parent
    print(f"Auto-detected DATA: {DATA}")
else:
    DATA = Path("/kaggle/input/birdclef-2026")
    print(f"WARNING: taxonomy.csv not found, using fallback: {DATA}")

# Model checkpoints: find any .pt file
_pt_matches = _glob.glob("/kaggle/input/**/*.pt", recursive=True)
if _pt_matches:
    MODEL_DIR = Path(_pt_matches[0]).parent
    print(f"Auto-detected MODEL_DIR: {MODEL_DIR}")
else:
    MODEL_DIR = Path("/kaggle/input/birdclef2026-efficientnet-b0")
    print(f"WARNING: no .pt files found, using fallback MODEL_DIR: {MODEL_DIR}")

# ---------------------------------------------------------------------------
# Runtime config
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STRIDE = 5.0  # non-overlapping 5s windows — aligns with submission row_id format

print(f"Device:    {DEVICE}")
print(f"DATA:      {DATA}")
print(f"MODEL_DIR: {MODEL_DIR}")

In [ ]:
# Cell 3 — Model definition and audio utilities (copied from train.py)

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------


class BirdModel(nn.Module):
    """EfficientNet (or any timm backbone) with a linear head for multilabel classification."""

    def __init__(self, backbone: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            backbone,
            pretrained=pretrained,
            num_classes=n_classes,
            in_chans=3,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


# ---------------------------------------------------------------------------
# Audio utilities
# ---------------------------------------------------------------------------


def make_melspec(y: np.ndarray, sr: int = SR) -> np.ndarray:
    """Compute log-mel spectrogram from a 1-D audio array."""
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
        fmin=FMIN,
        fmax=FMAX,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def spec_to_tensor(spec: np.ndarray) -> torch.Tensor:
    """Z-score normalise + stack to 3-channel tensor for ImageNet-pretrained backbone."""
    # spec: (n_mels, time) in dB, roughly [-80, 0]
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    img = np.stack([spec, spec, spec], axis=0)  # (3, n_mels, time)
    t = torch.from_numpy(img)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t - mean) / std


print("Model class and audio utilities defined.")

In [ ]:
# Cell 4 — Load species list and model checkpoints

# Species list: sorted primary_label from taxonomy.csv (234 species).
# This must match the order used during training.
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
species = sorted(taxonomy["primary_label"].astype(str).tolist())
n_species = len(species)
print(f"Species count: {n_species}")

# Find all checkpoint files
ckpt_paths = sorted(MODEL_DIR.glob("*.pt")) if MODEL_DIR.exists() else []
print(f"Checkpoints found: {len(ckpt_paths)} in {MODEL_DIR}")
for p in ckpt_paths:
    print(f"  {p.name}")

# Load each checkpoint into a BirdModel and set to eval mode
models = []
for ckpt_path in ckpt_paths:
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = BirdModel(backbone="efficientnet_b0", n_classes=n_species).to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    models.append(model)
    val_loss = ckpt.get("val_loss", float("nan"))
    epoch = ckpt.get("epoch", "?")
    print(f"  Loaded {ckpt_path.name} — epoch {epoch}, val_loss {val_loss:.4f}")

print(f"\nTotal models loaded: {len(models)}")
if len(models) == 0:
    print("WARNING: no models loaded — predictions will be zero-filled")

In [ ]:
# Cell 5 — Inference function


@torch.no_grad()
def predict_soundscape(
    path: Path,
    models: list,
    device: torch.device,
    stride: float = 5.0,
) -> np.ndarray:
    """Run sliding-window inference over a soundscape.

    Returns preds: np.ndarray of shape (n_chunks, n_species) with sigmoid
    probabilities averaged across all models, or zeros if no models loaded.
    """
    y, _ = librosa.load(path, sr=SR, mono=True)
    stride_samples = int(stride * SR)

    chunks = []
    pos = 0
    while pos + CLIP_SAMPLES <= len(y):
        chunk = y[pos : pos + CLIP_SAMPLES]
        spec = make_melspec(chunk)
        chunks.append(spec_to_tensor(spec))
        pos += stride_samples

    if not chunks:
        return np.zeros((1, n_species), dtype=np.float32)

    # No models loaded — return zeros
    if not models:
        return np.zeros((len(chunks), n_species), dtype=np.float32)

    batch = torch.stack(chunks).to(device)  # (n_chunks, 3, n_mels, time)

    all_preds = []
    for model in models:
        logits = model(batch)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

    return np.mean(all_preds, axis=0)  # (n_chunks, n_species)


print("predict_soundscape() defined.")

In [ ]:
# Cell 5b — Sanity checks
assert (DATA / "taxonomy.csv").exists(), f"taxonomy.csv not found at {DATA}"
assert (DATA / "sample_submission.csv").exists(), (
    f"sample_submission.csv not found at {DATA}"
)

ckpt_paths = sorted(MODEL_DIR.glob("*.pt")) if MODEL_DIR.exists() else []

tsd = DATA / "test_soundscapes"
soundscape_files = []
if tsd.exists():
    for ext in ("*.ogg", "*.wav", "*.flac"):
        soundscape_files.extend(tsd.rglob(ext))

print(f"DATA:             {DATA}")
print("Taxonomy:         OK")
print(f"Model checkpts:   {len(ckpt_paths)} .pt files in {MODEL_DIR}")
print(f"Test soundscapes: {len(soundscape_files)} audio files")
print(
    f"  All items in test_soundscapes: {sorted(f.name for f in tsd.iterdir()) if tsd.exists() else 'N/A'}"
)

# List all of /kaggle/input for debugging
print("\n/kaggle/input structure:")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_dir():
        print(f"  [DIR] {p.relative_to('/kaggle/input')}")

In [ ]:
# Cell 6 — Run inference and build submission.csv

# Load sample submission to learn expected row_ids and column order
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
print(f"Sample submission shape: {sample_sub.shape}")
print(sample_sub.head(3))

# Find test soundscapes
# NOTE: test_soundscapes/ is empty during interactive runs but populated by Kaggle at scoring time.
# We do NOT fall back to train_soundscapes — we align to sample_submission row_ids directly.
test_dir = DATA / "test_soundscapes"
soundscapes = []
for ext in ("*.ogg", "*.wav", "*.flac"):
    soundscapes.extend(sorted(test_dir.rglob(ext)))

print(f"Found {len(soundscapes)} test soundscapes in {test_dir}")

# Run inference per soundscape
rows = {}  # soundscape_stem -> list of (end_sec, preds)
for sc_path in soundscapes:
    print(f"  {sc_path.name} ...", end=" ", flush=True)
    preds = predict_soundscape(sc_path, models, DEVICE, stride=STRIDE)
    for i, chunk_preds in enumerate(preds):
        end_sec = (i + 1) * 5
        row_id = f"{sc_path.stem}_{end_sec}"
        rows[row_id] = chunk_preds
    print(f"{len(preds)} chunks")

print(f"Total rows from inference: {len(rows)}")

# Build submission aligned to sample_submission row order
expected_species = [c for c in sample_sub.columns if c != "row_id"]
out_rows = []
for _, r in sample_sub.iterrows():
    rid = r["row_id"]
    if rid in rows:
        preds = rows[rid]
        row = {"row_id": rid}
        for j, sp in enumerate(species):
            row[sp] = float(preds[j]) if j < len(preds) else 0.0
    else:
        # row_id not produced by inference (test soundscapes not available) — use zeros
        row = {"row_id": rid}
        for sp in species:
            row[sp] = 0.0
    out_rows.append(row)

sub = pd.DataFrame(out_rows)

# Fill any species columns missing from our model (zero-shot species)
for c in expected_species:
    if c not in sub.columns:
        sub[c] = 0.0

sub = sub[["row_id"] + expected_species]

# Save
out_path = Path("submission.csv")
sub.to_csv(out_path, index=False)
print(f"\nSubmission saved: {out_path}  shape={sub.shape}")
print(sub.head(5))